In [ ]:
import os
import json
from langchain.tools import tool
from supabase import create_client, Client
from dotenv import load_dotenv
from langchain_teddynote import logging

# .env 로드
load_dotenv()

# 프로젝트 이름을 입력합니다.
logging.langsmith("tzudong-storyboard-agent")

# Supabase 설정
SUPABASE_URL = os.getenv("PUBLIC_SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_SERVICE_ROLE_KEY")

if not SUPABASE_URL or not SUPABASE_KEY:
    print(
        "❌ 오류: .env 파일에 SUPABASE_URL 또는 SUPABASE_SERVICE_ROLE_KEY가 설정되지 않았습니다."
    )

In [ ]:
스토리보드 만든 후에 사용자 입력한 요구사항을 모두 다 적용되었는지 평가 yes or no -> 다시 쿼리짜서 RAG 또는 웹 검색 

검색된 자막 -> is_peak -> 프레임 캡셔닝이 최소 3개 장면 장면은 있어야 함 -> 그렇지 않으면, 사용자에게 추가적인 질문(human in the loop) -> 추가 RAG 또는 웹 검색



In [ ]:
from pydantic import BaseModel


class HumanRequest(BaseModel):
    """Forward the conversation to an expert. Use when you can't assist directly or the user needs assistance that exceeds your authority.
    To use this function, pass the user's 'request' so that an expert can provide appropriate guidance.
    """

    request: str

In [ ]:
from langchain import hub
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n\n".join(
        [
            f'<document><content>{doc.page_content}</content><source>{doc.metadata["source"]}</source><page>{doc.metadata["page"]+1}</page></document>'
            for doc in docs
        ]
    )


# RAG 체인 생성
rag_chain = prompt | llm | StrOutputParser()

# 체인 실행
generation = rag_chain.invoke({"context": format_docs(docs), "question": question})
print(generation)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser


llm = ChatOpenAI(model=MODEL_NAME, temperature=0)

# 시스템 프롬프트 정의
# 입력 질문을 벡터스토어 검색에 최적화된 형태로 변환하는 시스템 역할 정의
system = """You a question re-writer that converts an input question to a better version that is optimized \n 
     for vectorstore retrieval. Look at the input and try to reason about the underlying semantic intent / meaning."""

# 시스템 메시지와 초기 질문을 포함한 프롬프트 템플릿 생성
re_write_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        (
            "human",
            "Here is the initial question: \n\n {question} \n Formulate an improved question.",
        ),
    ]
)

# 질문 재작성기 생성
question_rewriter = re_write_prompt | llm | StrOutputParser()

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI


class GradeAnswer(BaseModel):
    """A binary score indicating whether the question is addressed."""

    # 답변의 관련성 평가: 'yes' 또는 'no'로 표기(yes: 관련성 있음, no: 관련성 없음)
    binary_score: str = Field(
        description="Answer addresses the question, 'yes' or 'no'"
    )


llm = ChatOpenAI(model=MODEL_NAME, temperature=0)

# llm 에 GradeAnswer 바인딩
structured_llm_grader = llm.with_structured_output(GradeAnswer)

# 시스템 프롬프트 정의
system = """You are a grader assessing whether an answer addresses / resolves a question \n 
     Give a binary score 'yes' or 'no'. Yes' means that the answer resolves the question."""

# 프롬프트 생성
answer_grader_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "User question: \n\n {question} \n\n LLM generation: {generation}"),
    ]
)

# 답변 평가기 생성
answer_grader = answer_grader_prompt | structured_llm_grader

In [ ]:
def router(query:str) -> str:    


In [ ]:
import operator
from typing import TypedDict, List, Annotated, Optional, Union
from typing_extensions import TypedDict
from langchain_openai import ChatOpenAI
from langchain_teddynote.tools.tavily import TavilySearch
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.messages import BaseMessage
from langchain_core.documents import Document


class AgentState(TypedDict):
    # 1. Base: Message History
    messages: Annotated[List[BaseMessage], operator.add]
    
    # 2. Control Flow & Mode
    mode: Literal['chat', 'qna', 'storyboard']  # Router가 설정 ("Q&A" vs "기획")
    next_step: str                               # Orchestrator 결정 ('tool', 'finish')
    loop_count: int                              # 무한 루프 방지
    retry_count: int                             # Query Refiner 재시도 횟수
    
    # 3. Context Data (Explicit)
    # Orchestrator가 수집한 핵심 문서들을 별도로 관리하여 Generator에 전달
    context_docs: Annotated[List[Document], operator.add]
    
    # 4. Query & Search State
    active_query: str                            # 현재 검색 중인 검색어
    search_results: Optional[str]                # 도구 실행 결과 임시 저장
    
    # 5. Validation & Feedback
    missing_info: Optional[str]                  # Validator가 지적한 부족한 점
    user_feedback: Optional[str]                 # Human Node에서 받은 사용자 입력
    
    # 6. Final Outputs
    final_answer: Optional[str]                  # Q&A 모드의 답변
    final_storyboard: Optional[dict]             # Storyboard 모드의 기획안



# 도구 초기화
tool = TavilySearch(max_results=3)


tools = [tool]

# LLM 초기화
llm = ChatOpenAI(model="gpt-4o-mini")

# 도구와 LLM 결합
llm_with_tools = llm.bind_tools(tools)


########## 3. 노드 추가 ##########
# 챗봇 함수 정의
def chatbot(state: State):
    # 메시지 호출 및 반환
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


# 상태 그래프 생성
graph_builder = StateGraph(State)

# 챗봇 노드 추가
graph_builder.add_node("chatbot", chatbot)

# ⭐️ 도구 노드 생성 및 추가 ⭐️ -> 이전 03에서는 Tool Node를 직접 구현했었음
tool_node = ToolNode(tools=[tool])

# ⭐️ tools 노드에 ToolNode 추가 ⭐️
graph_builder.add_node("tools", tool_node)

# tools > chatbot
graph_builder.add_edge("tools", "chatbot")

# START > chatbot
graph_builder.add_edge(START, "chatbot")

# chatbot > END
graph_builder.add_edge("chatbot", END)


# 조건부 엣지
graph_builder.add_conditional_edges(
    "chatbot",        # source에 chatbot 노드 (시작점)
    tools_condition,  # ⭐️ tools_condition: tools에서 호출할 것 있으면 호출하고, 없으면 END함 ⭐️
)

########## 5. MemorySaver 추가 ##########

# 메모리 저장소 초기화
memory = MemorySaver()

In [ ]:
# 전역 클라이언트 (연결 재사용)
_supabase_client = None


def get_supabase() -> Client:
    global _supabase_client
    if _supabase_client is None:
        _supabase_client = create_client(SUPABASE_URL, SUPABASE_KEY)
    return _supabase_client


@tool
def fetch_supabase_data(
    table_name: str, filter_column: str = None, filter_value: str = None, limit: int = 5
) -> str:
    """
    Supabase의 특정 테이블에서 데이터를 조회합니다.

    Args:
        table_name (str): 조회할 테이블 이름 (예: 'video_meta', 'restaurants', 'image_captions')
        filter_column (str): 필터링할 컬럼명 (선택사항, 예: 'title', 'id'). 없을 경우 전체 조회.
        filter_value (str): 필터링할 값 (선택사항, filter_column과 함께 사용). 단순 일치(eq) 조건.
        limit (int): 반환할 최대 행 수 (기본값: 5). 너무 많은 데이터를 가져오지 않도록 제한합니다.

    Returns:
        str: 조회된 데이터의 JSON 문자열. 에러 발생 시 에러 메시지 반환.
    """
    try:
        client = get_supabase()
        query = client.table(table_name).select("*")

        # 필터 조건이 있으면 적용
        if filter_column and filter_value:
            query = query.eq(filter_column, filter_value)

        # limit 적용 및 실행
        response = query.limit(limit).execute()

        if not response.data:
            return "[]"  # 빈 결과

        # 결과가 너무 길면 잘라서 보여주거나 요약할 수도 있지만, 일단 전체 반환
        return json.dumps(response.data, ensure_ascii=False)

    except Exception as e:
        return f"Error querying {table_name}: {str(e)}"

In [ ]:
# 테스트 실행
# result = fetch_supabase_data.invoke({"table_name": "video_meta", "limit": 1})
# print(result)